### Import the libraries


In [1]:
import pandas as pd
import numpy as np
import re
from scipy.spatial import cKDTree


### Load the raw data
Read the CSV file from the folder into a DataFrame called `df`.
`low_memory=False` tells pandas to look at the whole file before deciding each column's type, which
avoids misleading type warnings on a large file.

In [2]:
df=pd.read_csv('./work/StormEvents.csv',low_memory=False)

### Inspect the structure
`df.info()` lists every column, how many non-empty values it has, and its data type. Here we have ~1.83 million rows and 51 columns.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1830044 entries, 0 to 1830043
Data columns (total 51 columns):
 #   Column              Dtype  
---  ------              -----  
 0   BEGIN_YEARMONTH     int64  
 1   BEGIN_DAY           int64  
 2   BEGIN_TIME          int64  
 3   END_YEARMONTH       int64  
 4   END_DAY             int64  
 5   END_TIME            int64  
 6   EPISODE_ID          float64
 7   EVENT_ID            int64  
 8   STATE               str    
 9   STATE_FIPS          float64
 10  YEAR                int64  
 11  MONTH_NAME          str    
 12  EVENT_TYPE          str    
 13  CZ_TYPE             str    
 14  CZ_FIPS             int64  
 15  CZ_NAME             str    
 16  WFO                 str    
 17  BEGIN_DATE_TIME     str    
 18  CZ_TIMEZONE         str    
 19  END_DATE_TIME       str    
 20  INJURIES_DIRECT     int64  
 21  INJURIES_INDIRECT   int64  
 22  DEATHS_DIRECT       int64  
 23  DEATHS_INDIRECT     int64  
 24  DAMAGE_PROPERTY     str    
 25  DA

### Peek at the first rows
`df.head(10)` prints the first 10 records

In [4]:
df.head(10)

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,195004,28,1445,195004,28,1445,NaN,10096222,OKLAHOMA,40.0,...,NaN,NaN,NaN,35.12,-99.20,35.17,-99.20,NaN,NaN,PUB
1,195004,29,1530,195004,29,1530,NaN,10120412,TEXAS,48.0,...,NaN,NaN,NaN,31.90,-98.60,31.73,-98.60,NaN,NaN,PUB
2,195007,5,1800,195007,5,1800,NaN,10104927,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,40.58,-75.70,40.65,-75.47,NaN,NaN,PUB
3,195007,5,1830,195007,5,1830,NaN,10104928,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,40.60,-76.75,NaN,NaN,NaN,NaN,PUB
4,195007,24,1440,195007,24,1440,NaN,10104929,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,41.63,-79.68,NaN,NaN,NaN,NaN,PUB
5,195008,29,1600,195008,29,1600,NaN,10104930,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,40.22,-75.00,NaN,NaN,NaN,NaN,PUB
6,195011,4,1700,195011,4,1700,NaN,10104931,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,40.20,-76.12,40.27,-76.07,NaN,NaN,PUB
7,195011,4,1730,195011,4,1730,NaN,10104932,PENNSYLVANIA,42.0,...,NaN,NaN,NaN,40.27,-76.07,40.40,-75.93,NaN,NaN,PUB
8,195009,15,1745,195009,15,1745,NaN,10099490,OKLAHOMA,40.0,...,NaN,NaN,NaN,35.00,-96.25,35.07,-96.17,NaN,NaN,PUB
9,195009,16,130,195009,16,130,NaN,10099491,OKLAHOMA,40.0,...,NaN,NaN,NaN,34.83,-95.85,NaN,NaN,NaN,NaN,PUB


### Summarise the event-type column
For a text column, `describe()` reports how many values there are, how many are **distinct** (56
different event types), and the **most frequent** one (`Thunderstorm Wind`).

In [5]:
df['EVENT_TYPE'].describe().T

count               1830044
unique                   56
top       Thunderstorm Wind
freq                 514689
Name: EVENT_TYPE, dtype: object

### List every distinct event type
`unique()` returns all 56 different weather-event labels found in the data. Seeing them all is useful
before we group them into broader categories below.

In [6]:
df['EVENT_TYPE'].unique()

<StringArray>
[                   'Tornado',                       'Hail',
          'Thunderstorm Wind',                  'High Wind',
                'Flash Flood',               'Winter Storm',
                   'Blizzard',            'Cold/Wind Chill',
                 'Heavy Snow',                      'Flood',
                  'Ice Storm',                  'Dense Fog',
             'Winter Weather',                  'Avalanche',
               'Frost/Freeze',                  'Lightning',
                       'Heat',                 'Heavy Rain',
               'Funnel Cloud',              'Coastal Flood',
                'Strong Wind',                   'Wildfire',
                 'Waterspout',                  'High Surf',
                 'Dust Storm',                    'Drought',
                'Rip Current',                 'Dust Devil',
             'Tropical Storm',                'Debris Flow',
        'Hurricane (Typhoon)',               'Freezing Fog',
          

### Build a grouping dictionary
56 event types is a lot, and many are closely related. This **dictionary** maps each detailed type to
a smaller, broader category (for example `Hail` and `Lightning` both map to `Thunderstorm`). A
dictionary simply pairs each *key* (the original type) with a *value* (the group).

In [7]:
EVENT_GROUP_MAP = {
    # 1 · Tornado
    'Tornado': 'Tornado',
    'Funnel Cloud': 'Tornado',
    # 2 · Thunderstorm
    'Thunderstorm Wind': 'Thunderstorm',
    'Hail': 'Thunderstorm',
    'Lightning': 'Thunderstorm',
    'Heavy Rain': 'Thunderstorm',
    # 3 · Tropical cyclone
    'Hurricane (Typhoon)': 'Tropical_Cyclone',
    'Tropical Storm': 'Tropical_Cyclone',
    'Tropical Depression': 'Tropical_Cyclone',
    'Storm Surge/Tide': 'Tropical_Cyclone',
    'Marine Hurricane/Typhoon': 'Tropical_Cyclone',
    'Marine Tropical Storm': 'Tropical_Cyclone',
    'Marine Tropical Depression': 'Tropical_Cyclone',
    # 4 · Flooding
    'Flood': 'Flooding',
    'Flash Flood': 'Flooding',
    'Lakeshore Flood': 'Flooding',
    # 5 · Coastal flooding & surge
    'Coastal Flood': 'Coastal_Flood',
    'Astronomical Low Tide': 'Coastal_Flood',
    'Seiche': 'Coastal_Flood',
    'Sneakerwave': 'Coastal_Flood',
    'High Surf': 'Coastal_Flood',
    'Rip Current': 'Coastal_Flood',
    'Waterspout': 'Coastal_Flood',
    # 6 · Winter storm
    'Winter Storm': 'Winter_Storm',
    'Blizzard': 'Winter_Storm',
    'Heavy Snow': 'Winter_Storm',
    'Ice Storm': 'Winter_Storm',
    'Lake-Effect Snow': 'Winter_Storm',
    'Sleet': 'Winter_Storm',
    'Winter Weather': 'Winter_Storm',
    # 7 · Cold & freeze
    'Cold/Wind Chill': 'Cold',
    'Extreme Cold/Wind Chill': 'Cold',
    'Frost/Freeze': 'Cold',
    'Freezing Fog': 'Cold',
    # 8 · Extreme heat
    'Heat': 'Extreme_Heat',
    'Excessive Heat': 'Extreme_Heat',
    # 9 · Wildfire
    'Wildfire': 'Wildfire',
    'Dense Smoke': 'Wildfire',
    # 10 · Drought
    'Drought': 'Drought',
    # 11 · High wind
    'High Wind': 'High_Wind',
    'Strong Wind': 'High_Wind',
    'Marine High Wind': 'High_Wind',
    'Marine Strong Wind': 'High_Wind',
    'Marine Thunderstorm Wind': 'High_Wind',
    # 12 · Dust
    'Dust Storm': 'Dust',
    'Dust Devil': 'Dust',
    'Dense Fog': 'Dust',
    # 13 · Avalanche
    'Avalanche': 'Avalanche',
    'Debris Flow': 'Avalanche',
    # 14 · Volcanic
    'Volcanic Ash': 'Volcanic',
    'Volcanic Ashfall': 'Volcanic',
    # 15 · Tsunami
    'Tsunami': 'Tsunami',
    # 16 · Marine other
    'Marine Lightning': 'Marine_Other',
    'Marine Hail': 'Marine_Other',
    'Marine Dense Fog': 'Marine_Other',
    # 17 · Rare/geomagnetic
    'Northern Lights': 'Geomagnetic',
}



Copy df into df1. df remains a checkpoint we can fall back to.

In [8]:
df1=df.copy()

### Apply the grouping
`.map()` looks up each event type in the dictionary and writes the broad category into a new
`EVENT_GROUP` column. `.fillna('Other')` catches any type that wasn't in the dictionary and labels it
`'Other'`.

In [9]:
df1['EVENT_GROUP'] = df1['EVENT_TYPE'].map(EVENT_GROUP_MAP).fillna('Other')

### Check the result
Confirm the new column has collapsed the 56 detailed types into 17 broader groups.

In [10]:
df1['EVENT_GROUP'].unique()

<StringArray>
[         'Tornado',     'Thunderstorm',        'High_Wind',
         'Flooding',     'Winter_Storm',             'Cold',
             'Dust',        'Avalanche',     'Extreme_Heat',
    'Coastal_Flood',         'Wildfire',          'Drought',
 'Tropical_Cyclone',         'Volcanic',      'Geomagnetic',
     'Marine_Other',          'Tsunami']
Length: 17, dtype: str

### Summarise the groups
A quick summary of the new grouped column — `Thunderstorm` is the largest group (~949k events).

In [11]:
df1['EVENT_GROUP'].describe().T

count          1830044
unique              17
top       Thunderstorm
freq            949024
Name: EVENT_GROUP, dtype: object

### Measure the missing data (before cleaning)
For every column, count the empty cells and express it as a percentage of all rows. This map of the
gaps tells us what each cleaning step has to fix.

In [12]:
missing = df1.isnull().sum()/df1.shape[0]*100
missing

BEGIN_YEARMONTH        0.000000
BEGIN_DAY              0.000000
BEGIN_TIME             0.000000
END_YEARMONTH          0.000000
END_DAY                0.000000
END_TIME               0.000000
EPISODE_ID            12.690678
EVENT_ID               0.000000
STATE                  0.000055
STATE_FIPS             0.000055
YEAR                   0.000000
MONTH_NAME             0.000000
EVENT_TYPE             0.000000
CZ_TYPE                0.000000
CZ_FIPS                0.000000
CZ_NAME                0.085080
WFO                    6.862075
BEGIN_DATE_TIME        0.000000
CZ_TIMEZONE            0.000000
END_DATE_TIME          0.000000
INJURIES_DIRECT        0.000000
INJURIES_INDIRECT      0.000000
DEATHS_DIRECT          0.000000
DEATHS_INDIRECT        0.000000
DAMAGE_PROPERTY       31.632245
DAMAGE_CROPS          37.780348
SOURCE                18.897305
MAGNITUDE             42.529305
MAGNITUDE_TYPE        74.045487
FLOOD_CAUSE           94.385490
CATEGORY              99.972788
TOR_F_SC

### Spot malformed damage values
Damage is stored as **text** like `30K` (meaning 30,000). A few rows contain just a bare letter
`K`/`M`/`B` with **no number in front**, a data-entry error. This cell lists those rows so we know the
parser in the next step has to cope with them.

In [13]:
bad_damage_rows = df1[
    df1["DAMAGE_PROPERTY"].astype(str).str.strip().str.upper().isin(["K", "M", "B"]) |
    df1["DAMAGE_CROPS"].astype(str).str.strip().str.upper().isin(["K", "M", "B"])
]

bad_damage_rows[["DAMAGE_PROPERTY", "DAMAGE_CROPS", "EVENT_TYPE", "STATE", "YEAR"]]

,DAMAGE_PROPERTY,DAMAGE_CROPS,EVENT_TYPE,STATE,YEAR
387116,30K,K,Thunderstorm Wind,TEXAS,1999
390374,K,NaN,Strong Wind,WEST VIRGINIA,1999
441574,K,NaN,Thunderstorm Wind,ILLINOIS,2000
447213,10K,K,Thunderstorm Wind,TEXAS,2000
449859,100K,K,Hail,NEBRASKA,2000
463464,K,0,Flash Flood,FLORIDA,2000
479179,K,NaN,Wildfire,MONTANA,2001
491177,K,NaN,Flash Flood,MISSISSIPPI,2001
549966,NaN,K,Thunderstorm Wind,LOUISIANA,2002
562664,K,NaN,Drought,KANSAS,2002


### Define a damage parser
This function turns the text damage values into real numbers. It: trims spaces and upper-cases the
text; treats blanks / `NA` as missing (`NaN`); converts the suffix letter using multipliers
(`H`=100, `K`=1,000, `M`=1,000,000, `B`=1,000,000,000); and returns `NaN` when the value makes no
sense (such as a bare `K` with no number). Defining it here means we can reuse it on both damage
columns next.

In [14]:
def convert_to_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    if x in ["", "NA", "NAN", "NONE", "NULL"]:
        return np.nan

    # Clean obvious weird zero values
    if x in ["0?", "0T"]:
        return 0

    multipliers = {
        "H": 100,              # 5H = 500, 2H = 200
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000,
    }

    suffix = x[-1]

    if suffix in multipliers:
        number_part = x[:-1].strip()

        # Values like "K" or "M" have no number before the suffix
        if number_part == "":
            return np.nan

        try:
            return float(number_part) * multipliers[suffix]
        except ValueError:
            return np.nan

    try:
        return float(x)
    except ValueError:
        return np.nan

### Convert damage to numbers
First run the parser on both damage columns so they become numeric. Then fill any **still-missing**
damage with `0`: if no figure was recorded, we treat the monetary damage as zero. 

In [15]:
for col in ["DAMAGE_PROPERTY", "DAMAGE_CROPS"]:
    df1[col] = df1[col].apply(convert_to_number)

df1["DAMAGE_PROPERTY"] = df1["DAMAGE_PROPERTY"].fillna(0)
df1["DAMAGE_CROPS"] = df1["DAMAGE_CROPS"].fillna(0)
    


### Re-check the missing data
Confirm both damage columns now show 0% missing. The other gaps are still present and get handled by
the numbered steps below.

In [16]:
missing = df1.isnull().sum()/df1.shape[0]*100
missing

BEGIN_YEARMONTH        0.000000
BEGIN_DAY              0.000000
BEGIN_TIME             0.000000
END_YEARMONTH          0.000000
END_DAY                0.000000
END_TIME               0.000000
EPISODE_ID            12.690678
EVENT_ID               0.000000
STATE                  0.000055
STATE_FIPS             0.000055
YEAR                   0.000000
MONTH_NAME             0.000000
EVENT_TYPE             0.000000
CZ_TYPE                0.000000
CZ_FIPS                0.000000
CZ_NAME                0.085080
WFO                    6.862075
BEGIN_DATE_TIME        0.000000
CZ_TIMEZONE            0.000000
END_DATE_TIME          0.000000
INJURIES_DIRECT        0.000000
INJURIES_INDIRECT      0.000000
DEATHS_DIRECT          0.000000
DEATHS_INDIRECT        0.000000
DAMAGE_PROPERTY        0.000000
DAMAGE_CROPS           0.000000
SOURCE                18.897305
MAGNITUDE             42.529305
MAGNITUDE_TYPE        74.045487
FLOOD_CAUSE           94.385490
CATEGORY              99.972788
TOR_F_SC

### Start the main cleaning pass
Copy `df1` into `df2`. Every numbered cleaning step below works on `df2`, so `df1` remains a checkpoint
we can fall back to.

In [17]:
df2 = df1.copy()

print(f"Starting shape: {df2.shape}")
print(f"Columns: {list(df2.columns)}")


Starting shape: (1830044, 52)
Columns: ['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'EVENT_GROUP']


### Step 1 — Drop unhelpful columns
Remove columns that are almost entirely empty, exact duplicates of another column, or redundant with a cleaner field. 

In [18]:
# ============================================================
# STEP 1 — DROP COLUMNS
# Reason for each group:
#   CATEGORY              → officially never populated by NCEI (99.97% null)
#   TOR_OTHER_*           → only for rare cross-WFO tornadoes (99.81% null)
#   BEGIN/END decomposed  → fully redundant with BEGIN_DATE_TIME / END_DATE_TIME
#   STATE_FIPS            → deterministic duplicate of STATE
#   EPISODE_ID            → admin grouping key, not a predictive feature
#   BEGIN/END range+azim  → offset text and redundant
#                           with BEGIN_LAT/BEGIN_LON
#
# ============================================================

COLS_TO_DROP = [
    "CATEGORY",
    "TOR_OTHER_WFO", "TOR_OTHER_CZ_STATE", "TOR_OTHER_CZ_FIPS", "TOR_OTHER_CZ_NAME",
    "BEGIN_YEARMONTH", "BEGIN_DAY", "BEGIN_TIME",
    "END_YEARMONTH",   "END_DAY",   "END_TIME",
    "STATE_FIPS",
    "EPISODE_ID",
    "BEGIN_RANGE", "BEGIN_AZIMUTH",
    "END_RANGE",   "END_AZIMUTH",
]

df2 = df2.drop(columns=COLS_TO_DROP)

print(f"\nStep 1 — After drop: {df2.shape}  (removed {len(COLS_TO_DROP)} columns)")



Step 1 — After drop: (1830044, 35)  (removed 17 columns)


### Step 2 — Tornado intensity scale (`TOR_F_SCALE`)
Standardise the tornado rating: convert the old Fujita codes (`F0`–`F5`) to the modern
Enhanced-Fujita scale (`EF0`–`EF5`), as from documentation (Storm-Data-Bulk-csv-Format.pdf). Label a genuine tornado that has no rating as `'Unknown'`, and
label non-tornado events `'NA'` (not applicable).

In [19]:
# ============================================================
# STEP 2 — TOR_F_SCALE
#
# The column contains three kinds of values that need normalising:
#   - Legacy Fujita labels (F0–F5) from older records → map to EF equivalents
#   - 'EFU' (Enhanced Fujita Unknown) → treat as 'Unknown'
#   - NaN in Tornado events (108 rows) → fill 'Unknown'
#   - NaN in all other event types → fill 'NA' (not applicable)
# ============================================================

LEGACY_F = {
    "F0": "EF0", "F1": "EF1", "F2": "EF2",
    "F3": "EF3", "F4": "EF4", "F5": "EF5",
    "EFU": "Unknown",
}

tornado_mask = df2["EVENT_GROUP"] == "Tornado"

df2["TOR_F_SCALE"] = df2["TOR_F_SCALE"].replace(LEGACY_F)
df2.loc[~tornado_mask, "TOR_F_SCALE"] = df2.loc[~tornado_mask, "TOR_F_SCALE"].fillna("NA")
df2.loc[tornado_mask,  "TOR_F_SCALE"] = df2.loc[tornado_mask,  "TOR_F_SCALE"].fillna("Unknown")

assert df2["TOR_F_SCALE"].isna().sum() == 0, "TOR_F_SCALE still has nulls"
print(f"\nStep 2 — TOR_F_SCALE null: {df2['TOR_F_SCALE'].isna().sum()}")
print(f"  Value counts: {df2['TOR_F_SCALE'].value_counts().to_dict()}")





Step 2 — TOR_F_SCALE null: 0
  Value counts: {'NA': 1745470, 'EF0': 31296, 'EF1': 24900, 'Unknown': 12305, 'EF2': 11290, 'EF3': 3515, 'EF4': 1129, 'EF5': 139}


### Step 3 — Tornado length & width
These physical sizes only make sense for tornadoes. Non-tornado rows get `0.0`. A tornado missing a
value gets the **median** of the known tornado sizes.

In [20]:
# ============================================================
# STEP 3 — TOR_LENGTH / TOR_WIDTH
#
# Both are physical measurements that only apply to Tornado events.
# Non-tornado → 0.0 (the tornado does not exist, length/width = 0)
# Tornado with missing value → median of known tornado values
#   (median preferred over mean: tornado sizes are right-skewed)
# ============================================================

tor_len_med = df2.loc[tornado_mask & df2["TOR_LENGTH"].notna(), "TOR_LENGTH"].median()
tor_wid_med = df2.loc[tornado_mask & df2["TOR_WIDTH"].notna(),  "TOR_WIDTH"].median()

df2.loc[~tornado_mask, "TOR_LENGTH"] = df2.loc[~tornado_mask, "TOR_LENGTH"].fillna(0.0)
df2.loc[tornado_mask,  "TOR_LENGTH"] = df2.loc[tornado_mask,  "TOR_LENGTH"].fillna(tor_len_med)
df2.loc[~tornado_mask, "TOR_WIDTH"]  = df2.loc[~tornado_mask, "TOR_WIDTH"].fillna(0.0)
df2.loc[tornado_mask,  "TOR_WIDTH"]  = df2.loc[tornado_mask,  "TOR_WIDTH"].fillna(tor_wid_med)

assert df2["TOR_LENGTH"].isna().sum() == 0, "TOR_LENGTH still has nulls"
assert df2["TOR_WIDTH"].isna().sum()  == 0, "TOR_WIDTH still has nulls"
print(f"\nStep 3 — TOR_LENGTH null: {df2['TOR_LENGTH'].isna().sum()}  (tornado median: {tor_len_med})")
print(f"         TOR_WIDTH null:  {df2['TOR_WIDTH'].isna().sum()}   (tornado median: {tor_wid_med})")





Step 3 — TOR_LENGTH null: 0  (tornado median: 1.0)
         TOR_WIDTH null:  0   (tornado median: 50.0)


### Step 4 — Flood cause
Only flooding events have a cause. Non-flood rows are set to `'Not Applicable'` (which also tidies up a
few mislabeled rows), and flood rows with no recorded cause become `'Unknown'`.

In [21]:
# ============================================================
# STEP 4 — FLOOD_CAUSE
#
# Only applicable to Flooding events. There are 15 anomalous
# non-Flooding rows with a flood cause value — these are data
# entry errors. Overwriting the entire non-flood group with
# 'Not Applicable' corrects both the nulls and the anomalies.
# Missing Flooding rows → 'Unknown' (cause genuinely undocumented)
# ============================================================

flood_mask = df2["EVENT_GROUP"] == "Flooding"

df2.loc[~flood_mask, "FLOOD_CAUSE"] = "Not Applicable"
df2.loc[flood_mask,  "FLOOD_CAUSE"] = df2.loc[flood_mask, "FLOOD_CAUSE"].fillna("Unknown")

assert df2["FLOOD_CAUSE"].isna().sum() == 0, "FLOOD_CAUSE still has nulls"
print(f"\nStep 4 — FLOOD_CAUSE null: {df2['FLOOD_CAUSE'].isna().sum()}")
print(f"  Top values: {df2['FLOOD_CAUSE'].value_counts().head(4).to_dict()}")





Step 4 — FLOOD_CAUSE null: 0
  Top values: {'Not Applicable': 1672331, 'Heavy Rain': 92511, 'Unknown': 56294, 'Heavy Rain / Snow Melt': 4242}


### Step 5a — Recover `MAGNITUDE` from the narratives *(before estimating)*
Before falling back to group medians in Step 5, we first try to read the real value out of the
narrative text. Wind events often state the gust (`"gusts to 64 mph"`) and hail events the size
(`"1.75 inch hail"`). We extract those with gust-specific patterns only (so ambient numbers like
`"15 to 20 mph winds produced wind chill"` are ignored) and **convert mph → knots** because the
column is stored in knots. Anything the text can't supply is left as `NaN` and handled by the median
fill in Step 5.

In [22]:
# ============================================================
# STEP 5a — RECOVER MAGNITUDE FROM THE NARRATIVES (before estimation)
#
# A small but accurate gain: where a wind/hail event has no
# MAGNITUDE recorded but the narrative states it in prose
# ("gusts to 64 mph", "1.75 inch hail"), read the real value
# out of the text so the Step 5 median fill doesn't have to guess.
#
# Scope is deliberately narrow and TYPE-based (not group-based):
#   - true wind events  -> knots   (narratives are in MPH, so x0.868976)
#   - hail events       -> inches  (no conversion)
# Lightning / Heavy Rain are NOT touched here: their narratives may
# mention a gust, but that number is not this column's measurement.
#
# Only GUST-specific phrases are matched, to avoid grabbing ambient
# numbers like "15 to 20 mph winds produced wind chill". Validated
# against rows where MAGNITUDE is known: ~0.3 kt median error.
# Realistic yield is only a few thousand rows (most missing-MAGNITUDE
# rows are Flood / Winter / Drought events that have no magnitude at
# all and correctly become 0.0 in Step 5).
# ============================================================

GUST_RE = r'(?:gust(?:s|ing)?(?:\s+(?:of|to|up\s*to|near|around))?\s+|measured\s+(?:at\s+)?)(\d{2,3})\s*(mph|kt|knots?)'
ALT_RE  = r'(\d{2,3})\s*(mph|kt|knots?)\s+(?:wind\s+)?gust'
HAIL_RE = r'(\d(?:\.\d+)?)\s*inch'

WIND_TYPES = [
    "Thunderstorm Wind", "High Wind", "Strong Wind",
    "Marine High Wind", "Marine Strong Wind", "Marine Thunderstorm Wind",
]

# event-narrative first, then episode-narrative as context
narr = df2["EVENT_NARRATIVE"].fillna("") + " " + df2["EPISODE_NARRATIVE"].fillna("")

is_wind = df2["EVENT_TYPE"].isin(WIND_TYPES)                          # measured in knots
is_hail = df2["EVENT_TYPE"].str.contains("Hail", case=False, na=False) # measured in inches

before = int(df2["MAGNITUDE"].isna().sum())

# --- WIND: extract a gust, convert mph -> knots ---
w_mask = is_wind & df2["MAGNITUDE"].isna()
g   = narr[w_mask].str.extract(GUST_RE, flags=re.I)
alt = narr[w_mask].str.extract(ALT_RE,  flags=re.I)
val  = g[0].fillna(alt[0]).astype(float)
unit = g[1].fillna(alt[1]).str.lower()
val_kt = np.where(unit.eq("mph"), val * 0.868976, val)   # mph -> knots
df2.loc[w_mask, "MAGNITUDE"] = pd.Series(np.round(val_kt, 1), index=val.index)

# --- HAIL: extract size in inches (no conversion needed) ---
h_mask = is_hail & df2["MAGNITUDE"].isna()
hv = narr[h_mask].str.extract(HAIL_RE, flags=re.I)[0].astype(float)
df2.loc[h_mask, "MAGNITUDE"] = hv

after = int(df2["MAGNITUDE"].isna().sum())
print(f"\nStep 5a — MAGNITUDE recovered from narratives: {before - after:,}")
print(f"         Still missing (left for Step 5 median fill): {after:,}")



Step 5a — MAGNITUDE recovered from narratives: 4,021
         Still missing (left for Step 5 median fill): 774,284


### Step 5 — Magnitude and its type
Magnitude (wind speed in knots, or hail size in inches) mainly applies to thunderstorm / high-wind
events. Other events get `0.0` and `'NA'`. Missing wind/hail values are filled with that group's
**median** number and **most common** type label.

In [23]:
# ============================================================
# STEP 5 — MAGNITUDE / MAGNITUDE_TYPE
#
# These fields are populated primarily for Thunderstorm and
# High_Wind events (wind speed in knots) and for hail (inches).
# The 401 non-null values in other event groups are noise.
#
# Non-wind/hail events → MAGNITUDE = 0.0, MAGNITUDE_TYPE = 'NA'
# Missing wind/hail rows → group median for MAGNITUDE,
#                           group mode for MAGNITUDE_TYPE
# ============================================================

wind_hail_mask = df2["EVENT_GROUP"].isin(["Thunderstorm", "High_Wind"])

df2.loc[~wind_hail_mask, "MAGNITUDE"]      = df2.loc[~wind_hail_mask, "MAGNITUDE"].fillna(0.0)
df2.loc[~wind_hail_mask, "MAGNITUDE_TYPE"] = df2.loc[~wind_hail_mask, "MAGNITUDE_TYPE"].fillna("NA")

mag_med  = (
    df2.loc[wind_hail_mask & df2["MAGNITUDE"].notna()]
    .groupby("EVENT_GROUP")["MAGNITUDE"]
    .median()
)
mag_mode = (
    df2.loc[wind_hail_mask & df2["MAGNITUDE_TYPE"].notna()]
    .groupby("EVENT_GROUP")["MAGNITUDE_TYPE"]
    .agg(lambda x: x.mode().iloc[0])
)

for grp in mag_med.index:
    grp_mask = wind_hail_mask & (df2["EVENT_GROUP"] == grp)
    df2.loc[grp_mask, "MAGNITUDE"]      = df2.loc[grp_mask, "MAGNITUDE"].fillna(mag_med[grp])
    df2.loc[grp_mask, "MAGNITUDE_TYPE"] = df2.loc[grp_mask, "MAGNITUDE_TYPE"].fillna(mag_mode[grp])

assert df2["MAGNITUDE"].isna().sum()      == 0, "MAGNITUDE still has nulls"
assert df2["MAGNITUDE_TYPE"].isna().sum() == 0, "MAGNITUDE_TYPE still has nulls"
print(f"\nStep 5 — MAGNITUDE null:      {df2['MAGNITUDE'].isna().sum()}")
print(f"         MAGNITUDE_TYPE null: {df2['MAGNITUDE_TYPE'].isna().sum()}")
print(f"  Group medians: {mag_med.to_dict()}")
print(f"  Group modes:   {mag_mode.to_dict()}")





Step 5 — MAGNITUDE null:      0
         MAGNITUDE_TYPE null: 0
  Group medians: {'High_Wind': 50.0, 'Thunderstorm': 1.75}
  Group modes:   {'High_Wind': 'MG', 'Thunderstorm': 'EG'}


### Step 6 — Coordinates **and** location names (kept consistent)
`BEGIN_LOCATION` / `END_LOCATION` are kept for EDA / clustering, so the place names and the
coordinates must describe the **same** point, we fill them together so they can never contradict:

- **(a)** point events: copy the real start coords into the missing end coords;
- **(b)** if a coordinate is missing but the **location name is known**, take the coordinate from that
  location's median known position (location → coordinate);
- **(c)** any coordinate still missing → county/zone centroid → state → whole dataset, then copy the
  filled start coords into any missing end coords;
- **(d)** if a **location name is missing**, reverse-geocode it: assign the name of the nearest known
  point to the (now complete) coordinate, so the name always agrees with the coordinate. The rare
  residual with no coordinate at all becomes `'Unknown'`.

In [24]:
# ============================================================
# STEP 6 — GEOSPATIAL: COORDINATES + LOCATION NAMES (kept consistent)
#
# BEGIN_LOCATION / END_LOCATION are kept (useful for EDA / clustering),
# so the place name and the coordinate must always agree. We treat
# them as two views of the same point and fill them so they can never
# contradict each other:
#
#   6a. Point events: END coords <- BEGIN coords (preserve the real pair).
#   6b. Coordinate missing but LOCATION known -> coordinate taken from
#       that location's median known position (location -> coordinate).
#   6c. Coordinate still missing -> county/zone centroid -> state ->
#       global median; then propagate BEGIN coords into any missing END.
#   6d. LOCATION missing -> reverse-geocode the name of the nearest known
#       point to the (now complete) coordinate, so every imputed name
#       agrees with its coordinate by construction. Residual -> 'Unknown'.
# ============================================================

# --- 6a: point events (preserve real coordinate pairs) ---
point_mask = df2["END_LAT"].isna() & df2["BEGIN_LAT"].notna()
df2.loc[point_mask, "END_LAT"] = df2.loc[point_mask, "BEGIN_LAT"]
df2.loc[point_mask, "END_LON"] = df2.loc[point_mask, "BEGIN_LON"]
print(f"\nStep 6a — Filled {point_mask.sum()} point-event END coords from BEGIN")

# --- 6b: coordinate FROM the location name (location -> coordinate) ---
# Median known position of each named place (disambiguated by STATE).
loc_xy = (
    df2.dropna(subset=["BEGIN_LAT", "BEGIN_LON"])
    .groupby(["STATE", "BEGIN_LOCATION"])[["BEGIN_LAT", "BEGIN_LON"]]
    .median()
)

def _coord_from_loc(lat_col, lon_col, loc_col):
    need = df2[lat_col].isna() & df2[loc_col].notna()
    if not need.any():
        return 0
    idx = pd.MultiIndex.from_arrays([df2.loc[need, "STATE"], df2.loc[need, loc_col]])
    got = loc_xy.reindex(idx)
    df2.loc[need, lat_col] = got["BEGIN_LAT"].values
    df2.loc[need, lon_col] = got["BEGIN_LON"].values
    return int(got["BEGIN_LAT"].notna().sum())

n_b = _coord_from_loc("BEGIN_LAT", "BEGIN_LON", "BEGIN_LOCATION")
n_e = _coord_from_loc("END_LAT",   "END_LON",   "END_LOCATION")
print(f"Step 6b — Coords recovered from location name: BEGIN {n_b}, END {n_e}")

# --- 6c: centroid fallback for any coordinate still missing ---
cz_cent = (
    df2.dropna(subset=["BEGIN_LAT", "BEGIN_LON"])
    .groupby(["STATE", "CZ_TYPE", "CZ_FIPS"])[["BEGIN_LAT", "BEGIN_LON"]]
    .median().reset_index()
    .rename(columns={"BEGIN_LAT": "_lat_cz", "BEGIN_LON": "_lon_cz"})
)
state_cent = (
    df2.dropna(subset=["BEGIN_LAT", "BEGIN_LON"])
    .groupby("STATE")[["BEGIN_LAT", "BEGIN_LON"]]
    .median().reset_index()
    .rename(columns={"BEGIN_LAT": "_lat_st", "BEGIN_LON": "_lon_st"})
)
global_lat = df2["BEGIN_LAT"].median()
global_lon = df2["BEGIN_LON"].median()

df2 = df2.merge(cz_cent,    on=["STATE", "CZ_TYPE", "CZ_FIPS"], how="left")
df2 = df2.merge(state_cent, on="STATE", how="left")

df2["BEGIN_LAT"] = df2["BEGIN_LAT"].fillna(df2["_lat_cz"]).fillna(df2["_lat_st"]).fillna(global_lat)
df2["BEGIN_LON"] = df2["BEGIN_LON"].fillna(df2["_lon_cz"]).fillna(df2["_lon_st"]).fillna(global_lon)
df2 = df2.drop(columns=["_lat_cz", "_lon_cz", "_lat_st", "_lon_st"])

# propagate filled BEGIN coords into any remaining missing END coords
df2["END_LAT"] = df2["END_LAT"].fillna(df2["BEGIN_LAT"])
df2["END_LON"] = df2["END_LON"].fillna(df2["BEGIN_LON"])
print(f"Step 6c — Coord nulls after centroid fill: "
      f"BEGIN {df2['BEGIN_LAT'].isna().sum()}, END {df2['END_LAT'].isna().sum()}")

# --- 6d: reverse-geocode missing LOCATION from the final coordinates ---
def _reverse_geocode(loc_col, lat_col, lon_col):
    ref = df2[df2[loc_col].notna() & df2[lat_col].notna()]
    tree = cKDTree(ref[[lat_col, lon_col]].values)
    ref_loc = ref[loc_col].values
    need = df2[loc_col].isna() & df2[lat_col].notna()
    if need.any():
        _, ii = tree.query(df2.loc[need, [lat_col, lon_col]].values, k=1)
        df2.loc[need, loc_col] = ref_loc[ii]
    df2[loc_col] = df2[loc_col].fillna("Unknown")   # residual: no coordinate at all

_reverse_geocode("BEGIN_LOCATION", "BEGIN_LAT", "BEGIN_LON")
_reverse_geocode("END_LOCATION",   "END_LAT",   "END_LON")
print(f"Step 6d — Location nulls after reverse-geocode: "
      f"BEGIN {df2['BEGIN_LOCATION'].isna().sum()}, END {df2['END_LOCATION'].isna().sum()}")



Step 6a — Filled 177489 point-event END coords from BEGIN
Step 6b — Coords recovered from location name: BEGIN 66700, END 62198
Step 6c — Coord nulls after centroid fill: BEGIN 0, END 0
Step 6d — Location nulls after reverse-geocode: BEGIN 0, END 0


### Step 7 — Report source
Two fixes: make the capitalisation consistent (`.str.title()` makes `TRAINED SPOTTER` and
`Trained Spotter` count as the same source), then fill the missing sources with `'Unknown'`.

In [25]:
# ============================================================
# STEP 7 — SOURCE
#
# Two problems to fix:
#   1. Case inconsistency: "Trained Spotter" and "TRAINED SPOTTER"
#      are the same source. Apply .str.title() to unify.
#   2. 19.3% null: fill with 'Unknown'.
# ============================================================

df2["SOURCE"] = df2["SOURCE"].str.title().fillna("Unknown")

assert df2["SOURCE"].isna().sum() == 0, "SOURCE still has nulls"
print(f"\nStep 7 — SOURCE null: {df2['SOURCE'].isna().sum()}")
print(f"  Unique after normalisation: {df2['SOURCE'].nunique()} (was ~69 before)")





Step 7 — SOURCE null: 0
  Unique after normalisation: 57 (was ~69 before)


### Step 8 — Weather Forecast Office (`WFO`)
Fill a missing office with the most common (**mode**) office for that state. If a state has no office
information at all, fall back to `'Unknown'`.

In [26]:
# ============================================================
# STEP 8 — WFO
#
# 7.1% missing, concentrated in central/southern US states.
# Fill with the mode WFO per STATE. For states served by a
# single WFO this is exact; for multi-WFO states it picks
# the most common office in the sample.
# Final fallback 'Unknown' handles any state with no WFO
# data at all (unlikely in the full dataset).
# ============================================================

wfo_mode = (
    df2.loc[df2["WFO"].notna()]
    .groupby("STATE")["WFO"]
    .agg(lambda x: x.mode().iloc[0])
)

df2["WFO"] = (
    df2["WFO"]
    .fillna(df2["STATE"].map(wfo_mode))
    .fillna("Unknown")
)

assert df2["WFO"].isna().sum() == 0, "WFO still has nulls"
print(f"\nStep 8 — WFO null: {df2['WFO'].isna().sum()}")





Step 8 — WFO null: 0


### Step 9 — County / zone name
Only about 0.1% of these are missing, so we simply fill them with `'Unknown'`.

In [27]:
# ============================================================
# STEP 9 — CZ_NAME
#
# Only 0.11% missing (~21 rows). Fill with 'Unknown'.
# ============================================================

df2["CZ_NAME"] = df2["CZ_NAME"].fillna("Unknown")

assert df2["CZ_NAME"].isna().sum() == 0, "CZ_NAME still has nulls"
print(f"\nStep 9 — CZ_NAME null: {df2['CZ_NAME'].isna().sum()}")





Step 9 — CZ_NAME null: 0


### Fill the residual variables (`STATE` & `DATA_SOURCE`)

- **`STATE`** — 1 empty cell. It is a 2003 *Waterspout*, which happens over open water and was never
  assigned a state (its `STATE_FIPS` code was blank too).
- **`DATA_SOURCE`** — 15 empty cells. These are old *tornado* records from 1972–1996 that pre-date
  consistent source tagging.

There is no information inside the row to recover the true values, so we fill both with `'Unknown'`,
the same convention already used for `SOURCE`, `WFO` and `CZ_NAME`.

In [28]:
# ============================================================
# RESIDUAL VARIABLES — STATE & DATA_SOURCE
#
# The validation above flagged two columns that no earlier
# step touched, so their original nulls passed straight through:
#   STATE        ->  1 null  (a 2003 Waterspout over open water,
#                             never assigned a state; STATE_FIPS
#                             was blank too)
#   DATA_SOURCE  -> 15 nulls (legacy 1972-1996 tornado records,
#                             predating consistent source tagging)
#
# No in-row information can recover the true values, so both are
# filled with 'Unknown'
# ============================================================

df2["STATE"]       = df2["STATE"].fillna("Unknown")
df2["DATA_SOURCE"] = df2["DATA_SOURCE"].fillna("Unknown")

assert df2["STATE"].isna().sum() == 0, "STATE still has nulls"
assert df2["DATA_SOURCE"].isna().sum() == 0, "DATA_SOURCE still has nulls"

print(f"Residual fill — STATE null: {df2['STATE'].isna().sum()}, "
      f"DATA_SOURCE null: {df2['DATA_SOURCE'].isna().sum()}")

# Re-confirm: only the narrative text columns should remain null now.
remaining = df2.isnull().sum()
print("\nColumns still containing nulls (narratives are kept on purpose):")
print(remaining[remaining > 0])


Residual fill — STATE null: 0, DATA_SOURCE null: 0

Columns still containing nulls (narratives are kept on purpose):
EPISODE_NARRATIVE    479102
EVENT_NARRATIVE      856523
dtype: int64


### Validation 
Re-count the nulls. Everything should now be clean **except** the two free-text narrative columns

In [29]:
# ============================================================
# VALIDATION
# After all steps, the only allowed nulls are in the two
# narrative text columns (intentionally preserved for NLP).
# ============================================================

TEXT_COLS = ["EPISODE_NARRATIVE", "EVENT_NARRATIVE"]

print("\n" + "=" * 55)
print("FINAL MISSING VALUE REPORT")
print("=" * 55)

remaining = df2.isnull().sum()
remaining = remaining[remaining > 0]

non_text_nulls = remaining.drop(labels=[c for c in TEXT_COLS if c in remaining.index])

if len(non_text_nulls) == 0:
    print("Non-text columns: zero nulls ✓")
else:
    print("WARNING — unexpected nulls in non-text columns:")
    print(non_text_nulls)

print(f"\nNarrative nulls (expected):")
for col in TEXT_COLS:
    n = df2[col].isna().sum()
    pct = n / len(df2) * 100
    print(f"  {col}: {n} ({pct:.1f}%)")

print(f"\nFinal shape: {df2.shape}")
print(f"Remaining columns: {list(df2.columns)}")





FINAL MISSING VALUE REPORT
Non-text columns: zero nulls ✓

Narrative nulls (expected):
  EPISODE_NARRATIVE: 479102 (26.2%)
  EVENT_NARRATIVE: 856523 (46.8%)

Final shape: (1830044, 35)
Remaining columns: ['EVENT_ID', 'STATE', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'BEGIN_LOCATION', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'EVENT_GROUP']


### Missing & placeholder check
A plain `.isnull()` count is misleading now: several cleaning steps replaced real nulls with placeholder labels (`Unknown`, `NA`, `Not Applicable`). This check counts both true nulls and those placeholder strings together, per column, as a percentage of all rows — giving the real picture of how much of each column is non-informative right before the dataset is split and saved.

In [30]:
# ============================================================
# MISSING + PLACEHOLDER BREAKDOWN
#
# Build a DataFrame with one row per column and one column per
# "missing" category: true NaN plus each placeholder label
# introduced during cleaning, so we can see exactly which kind
# of missingness each column carries, instead of one merged number.
# ============================================================

PLACEHOLDER_MISSING = ["NA", "Unknown", "Not Applicable"]

missing_breakdown = pd.DataFrame({
    "NaN": df2.isnull().sum(),
    **{label: (df2 == label).sum() for label in PLACEHOLDER_MISSING}
})

missing_breakdown["Total"] = missing_breakdown.sum(axis=1)
missing_breakdown_pct = missing_breakdown / df.shape[0] * 100

missing_breakdown_pct

,NaN,NA,Unknown,Not Applicable,Total
EVENT_ID,0.000000,0.000000,0.000000,0.00000,0.000000
STATE,0.000000,0.000000,0.000055,0.00000,0.000055
YEAR,0.000000,0.000000,0.000000,0.00000,0.000000
MONTH_NAME,0.000000,0.000000,0.000000,0.00000,0.000000
EVENT_TYPE,0.000000,0.000000,0.000000,0.00000,0.000000
CZ_TYPE,0.000000,0.000000,0.000000,0.00000,0.000000
CZ_FIPS,0.000000,0.000000,0.000000,0.00000,0.000000
CZ_NAME,0.000000,0.000000,0.085080,0.00000,0.085080
WFO,0.000000,0.000000,0.000000,0.00000,0.000000
BEGIN_DATE_TIME,0.000000,0.000000,0.000000,0.00000,0.000000


### Step 10 — Split and save (3 structured files)
Write out 3 files: `StormEvents_cleaned_tabular.csv` keeps only numbers and categories
(the narratives are dropped) and is ready for machine learning;
`StormEvents_cleaned_tab_text.csv` keeps the narrative text and drops any rows that still have
gaps, for text-based analysis. 
`StormEvents_cleaned.csv` keeps the dataset as is.

In [32]:
# ============================================================
# STEP 10 — SPLIT AND SAVE
#============================================================

df_tabular = df2.drop(columns=TEXT_COLS)
df_tab_text    = df2.copy()
df_    = df2.copy()

# Drop rows where ANY of these columns have NaN/None
df_tab_text = df_tab_text.dropna()

# filter out rows where text columns are just empty strings or whitespace
for col in TEXT_COLS:
    df_tab_text = df_tab_text[df_tab_text[col].str.strip() != ""]


df_tabular.to_csv("./work/StormEvents_cleaned_tabular.csv", index=False)
df_tab_text.to_csv(   "./work/StormEvents_cleaned_tab_text.csv",    index=False)
df_.to_csv(   "./work/StormEvents_cleaned.csv",    index=False)


print("\n" + "=" * 55)
print("SAVED")
print("=" * 55)
print(f"df_tabular → StormEvents_cleaned_tabular.csv  shape: {df_tabular.shape}")
print(f"df_tab_text → StormEvents_cleaned_tab_text.csv     shape: {df_tab_text.shape}")
print(f"df_ → StormEvents_cleaned.csv     shape: {df_.shape}")




SAVED
df_tabular → StormEvents_cleaned_tabular.csv  shape: (1830044, 33)
df_tab_text → StormEvents_cleaned_tab_text.csv     shape: (810928, 35)
df_ → StormEvents_cleaned.csv     shape: (1830044, 35)
